# SquidPy spatial analysis of Leiden clusters

Automated spatial-analysis notebook for multiplex imaging AnnData.

Input: full `scimap_gated.h5ad` from the pipeline.  
Output: PDF plots only.

Main idea: treat the selected Leiden clustering as provisional cell-state/cell-type labels, then compute spatial graph statistics using SquidPy.

In [ ]:
# ==============================================================================
# PARAMETERS
# ==============================================================================

SAMPLE_INDEX = 1
SAMPLE_ID = "sample"
SAMPLE_DIR_NAME = "001_sample"

INPUT_PATH = "input_for_spatial_analysis.h5ad"
OUTPUT_DIR = "spatial_analysis_output"

CLUSTER_NORMALIZATION = "arcsinh"
CLUSTER_RESOLUTION = "0.5"

# Internal defaults; keep these stable unless you explicitly want to tune runtime.
SPATIAL_KEY = "spatial"
SQUIDPY_CLUSTER_KEY = "squidpy_cluster"

SQUIDPY_N_NEIGHBORS = 6
SQUIDPY_N_PERMUTATIONS = 100
SQUIDPY_N_JOBS = 1

RUN_COOCCURRENCE = True
COOCCURRENCE_INTERVALS = 30
COOCCURRENCE_MAX_CELLS = 100000
TOP_COOCCURRENCE_CLUSTERS = 8

RUN_SPATIAL_AUTOCORR = True
TOP_MARKERS_SPATIAL_AUTOCORR = 20

SPATIAL_PLOT_MAX_CELLS = 250000
PDF_DPI = 300

In [ ]:
# ==============================================================================
# IMPORTS AND LOAD FULL ANNDATA
# ==============================================================================

from pathlib import Path
import re
import gc
import warnings

import h5py
import numpy as np
import pandas as pd
import anndata as ad

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import squidpy as sq

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

INPUT_PATH = str(INPUT_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)

PLOT_DIR = OUTPUT_DIR / "plots"
SQUIDPY_PLOT_DIR = PLOT_DIR / "squidpy"
SQUIDPY_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility patch on the temporary local copy made by Nextflow.
# This does not touch the upstream SCIMAP AnnData.
with h5py.File(INPUT_PATH, "a") as f:
    if "uns/log1p" in f:
        del f["uns/log1p"]
        print("Patched /uns/log1p")

adata = ad.read_h5ad(INPUT_PATH)

print("=" * 80)
print("Loaded full AnnData")
print("=" * 80)
print("Sample:", SAMPLE_ID)
print("Input:", INPUT_PATH)
print("AnnData:", adata)
print("obs columns:", len(adata.obs.columns))
print("obsm keys:", list(adata.obsm.keys()))
print("layers:", list(adata.layers.keys()))
print("squidpy:", sq.__version__)
print("=" * 80)

In [ ]:
# ==============================================================================
# RESOLVE CLUSTER KEY AND FILTER CELLS
# ==============================================================================

def resolution_to_suffix(x):
    s = str(x).strip()
    s = s.replace(".", "_")
    return s

def safe_filename(x):
    s = str(x)
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def cluster_sort_key(x):
    s = str(x)
    if s.startswith("C"):
        s = s[1:]
    try:
        return (0, int(s))
    except Exception:
        try:
            return (0, float(s))
        except Exception:
            return (1, str(x))

cluster_resolution_suffix = resolution_to_suffix(CLUSTER_RESOLUTION)
cluster_key = f"leiden_{CLUSTER_NORMALIZATION}_res_{cluster_resolution_suffix}"
safe_cluster_key = safe_filename(cluster_key)

if cluster_key not in adata.obs.columns:
    available = [c for c in adata.obs.columns if str(c).startswith("leiden_")]
    raise KeyError(
        f"Requested cluster key not found: {cluster_key}\n"
        f"Available Leiden columns:\n" + "\n".join(map(str, available))
    )

if SPATIAL_KEY not in adata.obsm:
    raise KeyError(f"Missing adata.obsm[{SPATIAL_KEY!r}]. Available obsm keys: {list(adata.obsm.keys())}")

coords = np.asarray(adata.obsm[SPATIAL_KEY])
if coords.ndim != 2 or coords.shape[1] < 2:
    raise ValueError(f"Expected adata.obsm[{SPATIAL_KEY!r}] to have shape (n_cells, >=2), got {coords.shape}")

coords = coords[:, :2].astype(np.float32, copy=False)

cluster_series = adata.obs[cluster_key]
valid_cluster = cluster_series.notna().to_numpy()
valid_spatial = np.isfinite(coords).all(axis=1)

# Optional QC-column support if a previous notebook stored a boolean QC flag.
qc_mask = np.ones(adata.n_obs, dtype=bool)
qc_key_used = None
for candidate in [
    "qc_pass",
    "passed_qc",
    "pass_qc",
    "rapids_qc_pass",
    "cluster_qc_pass",
    "spatial_qc_pass",
]:
    if candidate in adata.obs.columns:
        vals = adata.obs[candidate]
        if vals.dropna().isin([True, False, 0, 1]).all():
            qc_mask = vals.fillna(False).astype(bool).to_numpy()
            qc_key_used = candidate
            break

mask = valid_cluster & valid_spatial & qc_mask

if mask.sum() == 0:
    raise ValueError("No cells remain after filtering for cluster label, spatial coordinates, and optional QC flag.")

adata_sq = adata[mask].copy()
coords_sq = coords[mask]

raw_labels = adata_sq.obs[cluster_key].astype(str)
cluster_counts = raw_labels.value_counts()
cluster_order = sorted(cluster_counts.index.tolist(), key=cluster_sort_key)

adata_sq.obs[SQUIDPY_CLUSTER_KEY] = pd.Categorical(
    raw_labels,
    categories=cluster_order,
)

adata_sq.obsm[SPATIAL_KEY] = coords_sq

print("=" * 80)
print("Filtered SquidPy input")
print("=" * 80)
print("Cluster key:", cluster_key)
print("Spatial key:", SPATIAL_KEY)
print("Optional QC key used:", qc_key_used)
print("Cells kept:", adata_sq.n_obs, "/", adata.n_obs)
print("Clusters:", len(cluster_order))
print(cluster_counts.loc[cluster_order].to_string())
print("=" * 80)

In [ ]:
# ==============================================================================
# PLOTTING HELPERS
# ==============================================================================

generated_pdfs = []

def save_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig = plt.gcf()
    fig.savefig(path, dpi=int(PDF_DPI), bbox_inches="tight")
    plt.close(fig)
    generated_pdfs.append(path)
    print("Saved:", path)

def plot_matrix_pdf(
    matrix,
    row_labels,
    col_labels,
    title,
    cbar_label,
    out_pdf,
    cmap="Blues",
    center_zero=False,
    annotate=False,
):
    arr = np.asarray(matrix, dtype=float)

    n_rows, n_cols = arr.shape
    fig_w = max(8, min(28, 4 + n_cols * 0.55))
    fig_h = max(7, min(26, 3 + n_rows * 0.45))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    if center_zero:
        vmax = np.nanmax(np.abs(arr))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        vmin = -vmax
    else:
        vmin = np.nanmin(arr)
        vmax = np.nanmax(arr)
        if not np.isfinite(vmin):
            vmin = 0.0
        if not np.isfinite(vmax) or vmax == vmin:
            vmax = vmin + 1.0

    im = ax.imshow(
        arr,
        cmap=cmap,
        aspect="auto",
        interpolation="nearest",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(title, fontsize=11, pad=12)
    ax.set_xlabel("Neighbor cluster")
    ax.set_ylabel("Source cluster")

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels([str(x) for x in col_labels], rotation=45, ha="right", fontsize=7)

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels([str(x) for x in row_labels], fontsize=7)

    if annotate and n_rows * n_cols <= 900:
        for i in range(n_rows):
            for j in range(n_cols):
                val = arr[i, j]
                if np.isfinite(val):
                    if abs(val) >= 100:
                        text = f"{val:.0f}"
                    elif abs(val) >= 10:
                        text = f"{val:.1f}"
                    else:
                        text = f"{val:.2f}"
                    ax.text(j, i, text, ha="center", va="center", fontsize=5)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label(cbar_label, fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    fig.tight_layout()
    fig.savefig(out_pdf, dpi=int(PDF_DPI), bbox_inches="tight")
    plt.close(fig)

    generated_pdfs.append(Path(out_pdf))
    print("Saved:", out_pdf)

    del fig, ax, im, arr
    gc.collect()

def plot_centrality_pdf(df, out_pdf):
    df = df.copy()
    df = df.loc[[c for c in cluster_order if c in df.index]]

    with PdfPages(out_pdf) as pdf:
        for col in df.columns:
            values = df[col].astype(float).sort_values(ascending=True)

            fig_h = max(5, min(18, 2 + len(values) * 0.35))
            fig, ax = plt.subplots(figsize=(9, fig_h))

            ax.barh(values.index.astype(str), values.values)
            ax.set_title(f"SquidPy centrality score: {col}\n{cluster_key}", fontsize=11)
            ax.set_xlabel(str(col))
            ax.set_ylabel("Cluster")
            ax.tick_params(axis="y", labelsize=7)

            fig.tight_layout()
            pdf.savefig(fig, dpi=int(PDF_DPI), bbox_inches="tight")
            plt.close(fig)

    generated_pdfs.append(Path(out_pdf))
    print("Saved:", out_pdf)

In [ ]:
# ==============================================================================
# SPATIAL MAP AND CLUSTER COUNTS
# ==============================================================================

# SquidPy official spatial scatter when possible; fallback to plain Matplotlib.
plot_adata = adata_sq

if plot_adata.n_obs > int(SPATIAL_PLOT_MAX_CELLS):
    rng = np.random.default_rng(0)

    selected = []
    labels = plot_adata.obs[SQUIDPY_CLUSTER_KEY].astype(str).to_numpy()

    counts = pd.Series(labels).value_counts()
    fractions = counts / counts.sum()

    for cl, frac in fractions.items():
        n_keep = max(1, int(frac * int(SPATIAL_PLOT_MAX_CELLS)))
        idx = np.where(labels == cl)[0]
        if len(idx) <= n_keep:
            selected.extend(idx.tolist())
        else:
            selected.extend(rng.choice(idx, size=n_keep, replace=False).tolist())

    selected = np.asarray(sorted(set(selected)), dtype=int)
    plot_adata = plot_adata[selected].copy()

print("Spatial map cells:", plot_adata.n_obs)

spatial_pdf = SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_squidpy_spatial_scatter.pdf"

try:
    sq.pl.spatial_scatter(
        plot_adata,
        spatial_key=SPATIAL_KEY,
        color=SQUIDPY_CLUSTER_KEY,
        shape=None,
        size=1,
        figsize=(9, 9),
        title=f"Spatial distribution of {cluster_key}",
        show=False,
    )
    save_fig(spatial_pdf)
except Exception as e:
    print("SquidPy spatial_scatter failed; using Matplotlib fallback.")
    print(type(e).__name__, str(e))

    xy = plot_adata.obsm[SPATIAL_KEY]
    labs = plot_adata.obs[SQUIDPY_CLUSTER_KEY].astype(str).to_numpy()
    cats = list(plot_adata.obs[SQUIDPY_CLUSTER_KEY].cat.categories)

    fig, ax = plt.subplots(figsize=(9, 9))
    for cl in cats:
        m = labs == cl
        ax.scatter(xy[m, 0], xy[m, 1], s=0.15, label=str(cl), rasterized=True)

    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(f"Spatial distribution of {cluster_key}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(markerscale=8, fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    save_fig(spatial_pdf)

counts_pdf = SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_cluster_counts.pdf"

counts = adata_sq.obs[SQUIDPY_CLUSTER_KEY].value_counts().loc[cluster_order].sort_values()
fig_h = max(5, min(18, 2 + len(counts) * 0.35))
fig, ax = plt.subplots(figsize=(9, fig_h))
ax.barh(counts.index.astype(str), counts.values)
ax.set_title(f"Cluster counts used for SquidPy analysis\n{cluster_key}")
ax.set_xlabel("Number of cells")
ax.set_ylabel("Cluster")
ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
save_fig(counts_pdf)

In [ ]:
# ==============================================================================
# SQUIDPY SPATIAL NEIGHBORS
# ==============================================================================

sq.gr.spatial_neighbors(
    adata_sq,
    spatial_key=SPATIAL_KEY,
    coord_type="generic",
    n_neighs=int(SQUIDPY_N_NEIGHBORS),
    key_added="spatial",
)

def force_writable_csr(adata_obj, key):
    """
    Some SquidPy/numba calls fail on HPC when scipy/joblib provides
    readonly CSR graph arrays. This makes the graph arrays writable.
    """
    if key not in adata_obj.obsp:
        return

    mat = adata_obj.obsp[key].tocsr(copy=True)

    mat.data = np.array(mat.data, copy=True)
    mat.indices = np.array(mat.indices, copy=True)
    mat.indptr = np.array(mat.indptr, copy=True)

    mat.data.setflags(write=True)
    mat.indices.setflags(write=True)
    mat.indptr.setflags(write=True)

    adata_obj.obsp[key] = mat

force_writable_csr(adata_sq, "spatial_connectivities")
force_writable_csr(adata_sq, "spatial_distances")

print("=" * 80)
print("SquidPy spatial graph")
print("=" * 80)
print("n_neighs:", SQUIDPY_N_NEIGHBORS)
print("n_jobs:", SQUIDPY_N_JOBS)
print("obsp keys:", [k for k in adata_sq.obsp.keys() if "spatial" in k])
print("uns keys:", [k for k in adata_sq.uns.keys() if "spatial" in k])
print("=" * 80)

In [ ]:
# ==============================================================================
# SQUIDPY NEIGHBORHOOD ENRICHMENT
# ==============================================================================

cats = list(adata_sq.obs[SQUIDPY_CLUSTER_KEY].cat.categories)

def fallback_graph_neighbor_counts():
    conn = adata_sq.obsp["spatial_connectivities"].tocsr(copy=True)
    coo = conn.tocoo()

    codes = adata_sq.obs[SQUIDPY_CLUSTER_KEY].cat.codes.to_numpy()
    n_cat = len(cats)

    counts = np.zeros((n_cat, n_cat), dtype=float)
    np.add.at(counts, (codes[coo.row], codes[coo.col]), 1)

    cluster_freq = np.bincount(codes, minlength=n_cat).astype(float)
    cluster_frac = cluster_freq / cluster_freq.sum()

    row_sum = counts.sum(axis=1)
    expected = row_sum[:, None] * cluster_frac[None, :]

    log2_enrichment = np.log2((counts + 1.0) / (expected + 1.0))

    return counts, log2_enrichment

try:
    force_writable_csr(adata_sq, "spatial_connectivities")
    force_writable_csr(adata_sq, "spatial_distances")

    sq.gr.nhood_enrichment(
        adata_sq,
        cluster_key=SQUIDPY_CLUSTER_KEY,
        connectivity_key="spatial_connectivities",
        n_perms=int(SQUIDPY_N_PERMUTATIONS),
        n_jobs=1,
        seed=0,
        show_progress_bar=True,
    )

    nhood_key = f"{SQUIDPY_CLUSTER_KEY}_nhood_enrichment"
    nhood = adata_sq.uns[nhood_key]

    zscore = np.asarray(nhood["zscore"], dtype=float)
    counts = np.asarray(nhood["count"], dtype=float)

    plot_matrix_pdf(
        zscore,
        row_labels=cats,
        col_labels=cats,
        title=f"SquidPy neighborhood enrichment z-score\n{cluster_key}",
        cbar_label="Enrichment z-score",
        out_pdf=SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_squidpy_nhood_enrichment_zscore.pdf",
        cmap="RdBu_r",
        center_zero=True,
        annotate=False,
    )

    plot_matrix_pdf(
        counts,
        row_labels=cats,
        col_labels=cats,
        title=f"SquidPy neighborhood enrichment observed counts\n{cluster_key}",
        cbar_label="Observed neighbor count",
        out_pdf=SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_squidpy_nhood_enrichment_counts.pdf",
        cmap="Blues",
        center_zero=False,
        annotate=True,
    )

except Exception as e:
    print("WARNING: SquidPy nhood_enrichment failed.")
    print("Using graph-based fallback from SquidPy spatial_connectivities.")
    print(type(e).__name__, str(e))

    counts, log2_enrichment = fallback_graph_neighbor_counts()

    plot_matrix_pdf(
        counts,
        row_labels=cats,
        col_labels=cats,
        title=f"Fallback neighbor counts from SquidPy graph\n{cluster_key}",
        cbar_label="Neighbor count",
        out_pdf=SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_fallback_graph_neighbor_counts.pdf",
        cmap="Blues",
        center_zero=False,
        annotate=True,
    )

    plot_matrix_pdf(
        log2_enrichment,
        row_labels=cats,
        col_labels=cats,
        title=f"Fallback log2 neighbor enrichment from SquidPy graph\n{cluster_key}",
        cbar_label="log2 observed/expected",
        out_pdf=SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_fallback_graph_log2_neighbor_enrichment.pdf",
        cmap="RdBu_r",
        center_zero=True,
        annotate=False,
    )

In [ ]:
# ==============================================================================
# SQUIDPY INTERACTION MATRIX
# ==============================================================================

cats = list(adata_sq.obs[SQUIDPY_CLUSTER_KEY].cat.categories)

try:
    force_writable_csr(adata_sq, "spatial_connectivities")

    interaction_counts = sq.gr.interaction_matrix(
        adata_sq,
        cluster_key=SQUIDPY_CLUSTER_KEY,
        connectivity_key="spatial_connectivities",
        normalized=False,
        copy=True,
    )

    interaction_norm = sq.gr.interaction_matrix(
        adata_sq,
        cluster_key=SQUIDPY_CLUSTER_KEY,
        connectivity_key="spatial_connectivities",
        normalized=True,
        copy=True,
    )

except Exception as e:
    print("WARNING: SquidPy interaction_matrix failed.")
    print("Using graph-based fallback from SquidPy spatial_connectivities.")
    print(type(e).__name__, str(e))

    conn = adata_sq.obsp["spatial_connectivities"].tocsr(copy=True)
    coo = conn.tocoo()

    codes = adata_sq.obs[SQUIDPY_CLUSTER_KEY].cat.codes.to_numpy()
    n_cat = len(cats)

    interaction_counts = np.zeros((n_cat, n_cat), dtype=float)
    np.add.at(interaction_counts, (codes[coo.row], codes[coo.col]), 1)

    row_sums = interaction_counts.sum(axis=1, keepdims=True)
    interaction_norm = np.divide(
        interaction_counts,
        row_sums,
        out=np.zeros_like(interaction_counts),
        where=row_sums > 0,
    )

plot_matrix_pdf(
    interaction_counts,
    row_labels=cats,
    col_labels=cats,
    title=f"Interaction matrix: edge counts\n{cluster_key}",
    cbar_label="Graph edge count",
    out_pdf=SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_interaction_matrix_counts.pdf",
    cmap="Blues",
    center_zero=False,
    annotate=True,
)

plot_matrix_pdf(
    interaction_norm,
    row_labels=cats,
    col_labels=cats,
    title=f"Interaction matrix: row-normalized\n{cluster_key}",
    cbar_label="Row-normalized edge fraction",
    out_pdf=SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_interaction_matrix_normalized.pdf",
    cmap="Blues",
    center_zero=False,
    annotate=True,
)

In [ ]:
# ==============================================================================
# SQUIDPY CENTRALITY SCORES
# ==============================================================================

try:
    force_writable_csr(adata_sq, "spatial_connectivities")

    centrality = sq.gr.centrality_scores(
        adata_sq,
        cluster_key=SQUIDPY_CLUSTER_KEY,
        connectivity_key="spatial_connectivities",
        copy=True,
        n_jobs=1,
    )

except Exception as e:
    print("WARNING: SquidPy centrality_scores failed.")
    print("Using fallback mean graph degree per cluster.")
    print(type(e).__name__, str(e))

    conn = adata_sq.obsp["spatial_connectivities"].tocsr(copy=True)
    degree = np.asarray(conn.sum(axis=1)).ravel()

    labels = adata_sq.obs[SQUIDPY_CLUSTER_KEY].astype(str).to_numpy()

    centrality = (
        pd.DataFrame({"cluster": labels, "mean_graph_degree": degree})
        .groupby("cluster")
        .agg(
            mean_graph_degree=("mean_graph_degree", "mean"),
            median_graph_degree=("mean_graph_degree", "median"),
        )
    )

print("Centrality scores:")
display(centrality)

centrality_pdf = SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_centrality_scores.pdf"
plot_centrality_pdf(centrality, centrality_pdf)

In [ ]:
# ==============================================================================
# SQUIDPY CO-OCCURRENCE
# ==============================================================================

def stratified_subset_by_cluster(adata_in, cluster_key, max_cells, seed=0):
    if adata_in.n_obs <= int(max_cells):
        return adata_in.copy()

    rng = np.random.default_rng(seed)
    labels = adata_in.obs[cluster_key].astype(str).to_numpy()
    counts = pd.Series(labels).value_counts()
    fractions = counts / counts.sum()

    selected = []

    for cl, frac in fractions.items():
        n_keep = max(1, int(frac * int(max_cells)))
        idx = np.where(labels == cl)[0]

        if len(idx) <= n_keep:
            selected.extend(idx.tolist())
        else:
            selected.extend(rng.choice(idx, size=n_keep, replace=False).tolist())

    selected = np.asarray(sorted(set(selected)), dtype=int)
    return adata_in[selected].copy()

if RUN_COOCCURRENCE:

    try:
        adata_co = stratified_subset_by_cluster(
            adata_sq,
            SQUIDPY_CLUSTER_KEY,
            max_cells=int(COOCCURRENCE_MAX_CELLS),
            seed=0,
        )

        print("=" * 80)
        print("SquidPy co-occurrence input")
        print("=" * 80)
        print("Cells:", adata_co.n_obs, "/", adata_sq.n_obs)
        print("Intervals:", COOCCURRENCE_INTERVALS)
        print("=" * 80)

        sq.gr.co_occurrence(
            adata_co,
            cluster_key=SQUIDPY_CLUSTER_KEY,
            spatial_key=SPATIAL_KEY,
            interval=int(COOCCURRENCE_INTERVALS),
            n_jobs=1,
        )

        co_key = f"{SQUIDPY_CLUSTER_KEY}_co_occurrence"
        co_data = adata_co.uns[co_key]

        occ = np.asarray(co_data["occ"], dtype=float)
        interval = np.asarray(co_data["interval"], dtype=float)
        cats_co = list(adata_co.obs[SQUIDPY_CLUSTER_KEY].cat.categories)

        if interval.size == occ.shape[-1] + 1:
            x = interval[1:]
        else:
            x = interval[: occ.shape[-1]]

        top_conditional_clusters = (
            adata_co.obs[SQUIDPY_CLUSTER_KEY]
            .value_counts()
            .head(int(TOP_COOCCURRENCE_CLUSTERS))
            .index
            .astype(str)
            .tolist()
        )

        co_pdf = SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_squidpy_co_occurrence_top{TOP_COOCCURRENCE_CLUSTERS}.pdf"

        with PdfPages(co_pdf) as pdf:
            for cond in top_conditional_clusters:
                if cond not in cats_co:
                    continue

                cond_idx = cats_co.index(cond)

                fig, ax = plt.subplots(figsize=(9, 5.5))

                if occ.ndim == 3 and occ.shape[0] == len(cats_co) and occ.shape[1] == len(cats_co):
                    curves = occ[cond_idx, :, :]

                    for obs_idx, obs_cluster in enumerate(cats_co):
                        ax.plot(
                            x,
                            curves[obs_idx, : len(x)],
                            linewidth=1.2,
                            label=str(obs_cluster),
                        )

                ax.axhline(1.0, linestyle="--", linewidth=0.8)
                ax.set_title(f"SquidPy co-occurrence around cluster {cond}\n{cluster_key}")
                ax.set_xlabel("Distance bin")
                ax.set_ylabel("Co-occurrence probability ratio")
                ax.legend(
                    title="Observed cluster",
                    fontsize=6,
                    title_fontsize=7,
                    ncol=2,
                    frameon=False,
                )
                fig.tight_layout()
                pdf.savefig(fig, dpi=int(PDF_DPI), bbox_inches="tight")
                plt.close(fig)

        generated_pdfs.append(co_pdf)
        print("Saved:", co_pdf)

    except Exception as e:
        print("WARNING: SquidPy co_occurrence failed; continuing.")
        print(type(e).__name__, str(e))

        fail_pdf = SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_squidpy_co_occurrence_failed.pdf"
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis("off")
        ax.text(
            0.02,
            0.95,
            "SquidPy co-occurrence failed, but the notebook continued.\n\n"
            f"Error type: {type(e).__name__}\n"
            f"Error message: {str(e)[:700]}",
            va="top",
            ha="left",
            fontsize=9,
            family="monospace",
            transform=ax.transAxes,
        )
        fig.tight_layout()
        fig.savefig(fail_pdf, dpi=int(PDF_DPI), bbox_inches="tight")
        plt.close(fig)

        generated_pdfs.append(fail_pdf)
        print("Saved:", fail_pdf)

else:
    print("Skipping co-occurrence.")

In [ ]:
# ==============================================================================
# SQUIDPY SPATIAL AUTOCORRELATION OF MARKERS
# ==============================================================================

if RUN_SPATIAL_AUTOCORR and adata_sq.n_vars > 0:
    marker_names = list(map(str, adata_sq.var_names))
    marker_names = marker_names[: int(TOP_MARKERS_SPATIAL_AUTOCORR)]

    print("=" * 80)
    print("Running SquidPy spatial autocorrelation")
    print("=" * 80)
    print("Markers:", marker_names)
    print("=" * 80)

    try:
        sq.gr.spatial_autocorr(
            adata_sq,
            connectivity_key="spatial_connectivities",
            genes=marker_names,
            mode="moran",
            n_perms=None,
            n_jobs=1,
        )

        if "moranI" in adata_sq.uns:
            moran = adata_sq.uns["moranI"].copy()
        elif "moran" in adata_sq.uns:
            moran = adata_sq.uns["moran"].copy()
        else:
            moran = None

        if moran is not None:
            # SquidPy returns a DataFrame indexed by gene/marker.
            score_col = None
            for candidate in ["I", "moranI", "score"]:
                if candidate in moran.columns:
                    score_col = candidate
                    break

            if score_col is None:
                numeric_cols = moran.select_dtypes(include=[np.number]).columns.tolist()
                score_col = numeric_cols[0] if numeric_cols else None

            if score_col is not None:
                moran_plot = moran.sort_values(score_col, ascending=True)

                fig_h = max(5, min(14, 2 + len(moran_plot) * 0.35))
                fig, ax = plt.subplots(figsize=(9, fig_h))
                ax.barh(moran_plot.index.astype(str), moran_plot[score_col].astype(float).values)
                ax.set_title(f"SquidPy Moran's I marker spatial autocorrelation\n{cluster_key}")
                ax.set_xlabel("Moran's I")
                ax.set_ylabel("Marker")
                ax.tick_params(axis="y", labelsize=7)
                fig.tight_layout()

                moran_pdf = SQUIDPY_PLOT_DIR / f"{safe_cluster_key}_squidpy_marker_morans_i.pdf"
                fig.savefig(moran_pdf, dpi=int(PDF_DPI), bbox_inches="tight")
                plt.close(fig)

                generated_pdfs.append(moran_pdf)
                print("Saved:", moran_pdf)

            display(moran)

        else:
            print("Could not find Moran result in adata.uns.")

    except Exception as e:
        print("Spatial autocorrelation failed; continuing.")
        print(type(e).__name__, str(e))

else:
    print("Skipping spatial autocorrelation.")

In [ ]:
# ==============================================================================
# FINAL OUTPUT LIST
# ==============================================================================

print("=" * 80)
print("Generated PDF outputs")
print("=" * 80)

for path in generated_pdfs:
    print(" -", path)

print("=" * 80)
print("No CSV files are written by this notebook.")
print("=" * 80)